# Getting Started

In this example we will go over the basics of Tierkreis workflows:
1. Setting up Tierkreis
2. Defining a graph through its inputs and outputs
3. Constructing the computation
    - Using the inputs
    - Using simple nodes
    - Using builting functionality
4. Having a first look at the visualizer
5. Running the graph
6. Creating an entire project

This example a simplified version of the sample given in the project initialization.
We will have a look at setting up an entire project at the end of this example

## Setting up Tierkreis

Tierkreis works best with the [uv package manager](https://docs.astral.sh/uv/). We strongly recommend using it as your package manager for Tierkreis projects.

To get started with Tierkreis start a new `uv` project in an empty directory with:

```bash
uv init
```

Then add Tierkreis to the project and run the project setup tool.

```bash
uv add tierkreis
```

## Defining the first graph

First we instantiate a `GraphBuilder` object.
It is the main interface to construct graphs.
The arguments to the constructor describe the inputs and outputs of the graph respectively.
The following graph takes one string type input and outputs a single string.
In general a graph can have a multiple inputs and multiple outputs
but for now we keep things simple.

```{note}
In order to keep a clear separation between the types used in the Tierkreis graph and the types already present in the Python language we wrap the former with `TKR`.
(The `TKR[A]` wrapper type indicates that an edge in the graph contains a value of type `A`. More on this on the bottom of the Pate)
```

In [ ]:
from tierkreis.builder import GraphBuilder
from tierkreis.controller.data.models import TKR

hello_world_graph = GraphBuilder(TKR[str], TKR[str])

## Constructing the Graph

Next we're going to construct a graph by calling the builder functions.
Each function adds a node to the graph, keeping track of the necessary edges of values provided to the node.

We can add constants to a graph using a `GraphBuilder.const` node which doesn't take inputs, but instead holds a single value.

In [ ]:
const = hello_world_graph.const("Hello ")

The `const` variable now holds a reference to the constant value `Hello ` which we now can use as inputs to further nodes by passing the python variable.
Similarly we can address workflow input values (internally they are also nodes) by addressing `hello_world_graph.inputs`. 
We're going to concatenate `const` string and the input string using a `GraphBuilder.task`.
Tasks provide additional functionality that are treated as atomic operation.
In this case, we're using a simple string concatenation, which is available as a `builtin` task.
As inputs it takes two strings and will produce one string output.



In [ ]:
from tierkreis.builtins import concat

out = hello_world_graph.task(concat(const, hello_world_graph.inputs))

To finish, we specify the outputs of the graph.

In [ ]:
hello_world_graph.outputs(out)

## Using the visualizer

Tierkreis comes with an additional library to keep track of your workflows.
You can also use it to examine graphs that you are currently constructing.
To use it, install
```bash
uv add tierkreis-visualizer
```
The visualizer will run a local web application in the same process.
To stop its execution you need to user `ctrl+c`. 

In [ ]:
from tierkreis_visualization.visualize_graph import visualize_graph

visualize_graph(hello_world_graph)

In the next tutorial we will give a more detailed explanation of the visualizer and how to use it.

## Running the graph

To run a general Tierkreis graph we need to set up:

- a way to store and share input and output values (the 'storage' interface)
- a way to run tasks (the 'executor' interface)

For this example we use the `FileStorage` that is provided by the Tierkreis library itself.
The inputs and outputs will be stored in a directory on disk.
(By default the files are stored in `~/.tierkreis/checkpoints/<WORKFLOW_ID>`, where `<WORKFLOW_ID>` is a `UUID` identifying the workflow.)

In [ ]:
from uuid import UUID


from tierkreis.storage import FileStorage

storage = FileStorage(workflow_id=UUID(int=12345), name="Hello World Graph")

If we have already run this example then there will already be files at this directory in the storage.
If we want to reuse the directory then run

In [ ]:
storage.clean_graph_files()

to get a fresh area to work in.

Since we are just using the Tierkreis built-in tasks the executor will not actually be called.
As a placeholder we create a simple `ShellExecutor`, also provided by the Tierkreis library, which can run bash scripts in a specified directory. In this case we can use `None`.


In [ ]:
from tierkreis.executor import ShellExecutor

executor = ShellExecutor(registry_path=None, workflow_dir=storage.workflow_dir)

As the penultimate step we need to provide the workflow inputs to run as a dictionary.
If the inputs are not provided the workflow will encounter an error.
In our example the workflow expects a string input called `value` (the default name for unnamed inputs).

In [ ]:
inputs = {"value": "world!"}


With the storage and executor specified and inputs set, we can now run a graph using `run_graph`.

In [ ]:
from tierkreis.controller import run_graph
from tierkreis.storage import read_outputs

run_graph(storage, executor, hello_world_graph.get_data(), inputs)
result = read_outputs(hello_world_graph, storage)
print(result)


## On tierkreis types

Tierkreis values correspond to the edges in the graph.
These values can have a type assigned to them at construction time.
The set of available types is a subset of all python types, e.g. we require serialization; see more in [complex types](../worker/complex_types.md) how to add your own serialization.
As result we use the `TKR` container to promote the python types into tierkreis compatible types.